# StealTheDealAI: Fine-Tune Llama-3.2-3B (QLoRA, Kaggle T4)

Fine-tunes `meta-llama/Llama-3.2-3B` with QLoRA on the price-prediction JSONL and pushes the
LoRA adapter to the Hugging Face Hub, ready for the Modal deployment.

## Before you run

1. **Add-ons → Secrets** → add `HF_TOKEN` (a Hugging Face **write** token).
2. Turn on a **GPU** accelerator.
3. Attach the dataset containing `finetune_train.jsonl` and `finetune_val.jsonl`.
4. **Set `HF_USER`** in the imports cell to your Hugging Face username (it's asserted there,
   so a placeholder fails immediately rather than after hours of training).
5. **Run → Restart Session, then Run All.** Some fixes below (`CUDA_VISIBLE_DEVICES`, the
   `pip install` without `-U`) only take effect in a fresh process — re-running cells in an
   already-initialized kernel makes them silent no-ops and the old errors reappear.

Expect roughly **2–3 hours** for 3 epochs on a T4. Well inside Kaggle's session limit, but
plan the GPU quota accordingly.

## Errors this notebook has already been fixed for

Each of these was hit for real on Kaggle and traced to root cause:

| Symptom | Root cause | Fix |
|---|---|---|
| Imports break outright | `pip install -U` upgrades torch/transformers/peft/accelerate independently into a mutually-incompatible set | `pip install` **without** `-U` — keep Kaggle's pre-tested base versions |
| `CUBLAS_STATUS_EXECUTION_FAILED` inside a `DataParallel` replica | `Trainer` auto-wraps in `torch.nn.DataParallel` when >1 GPU is visible (independent of `device_map`); bitsandbytes 4-bit layers don't survive its replication | `CUDA_VISIBLE_DEVICES="0"` **before any import** |
| `'functools.partial' object has no attribute '__func__'` | TRL's newer default `loss_type="chunked_nll"` monkey-patches the model's forward; incompatible with a PeftModel-over-4bit stack | `loss_type="nll"` — identical math, no patching |
| `"_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'` | `fp16=True` uses a `GradScaler` that can only unscale fp16/fp32 grads. Gradient dtype follows **parameter** dtype, and whether the LoRA adapters land in fp32 depends on `prepare_model_for_kbit_training()`'s upcasting, which changed across PEFT versions — so setting the *base model's* dtype does not control it | Explicitly recast **every trainable param to fp32** after `get_peft_model()`, with an `assert` to prove it |
| `warmup_ratio is deprecated` | Removed in transformers v5.2 | `warmup_steps`, computed from the real dataset size |
| `You passed a PeftModel instance together with a peft_config` | Model is already wrapped by `get_peft_model()`; passing `peft_config` again double-applies it | Don't pass `peft_config=` to `SFTTrainer` |
| — | `tokenizer=` deprecated; `max_seq_length=` moved off `SFTTrainer` | `processing_class=`; `max_length=` on `SFTConfig` |

The `"tokenizer has new PAD/BOS/EOS tokens … config were aligned accordingly"` message is
**benign** — transformers just noting it synced `pad_token_id` after we set
`tokenizer.pad_token = tokenizer.eos_token`. Not an error.

## Training choices

- **fp16 throughout, never bf16.** T4 (compute capability 7.5) and P100 (6.0) have no native
  bf16; `SFTConfig` defaults `bf16=True` when `fp16` is unset, so both are set explicitly.
- **3 epochs**, not a fixed step count. The original `max_steps=500` was under 0.4 epochs on
  the real ~23k-row training set. `load_best_model_at_end=True` pushes the lowest-`eval_loss`
  checkpoint, so overshooting slightly is harmless.
- **Loss on the completion only.** The JSONL is already `{"prompt", "completion"}`, which TRL
  handles natively — no `formatting_func`, so no gradient is wasted on the fixed question text.
- **Eval subsampled to 800 rows** — it only ranks checkpoints, and evaluating all ~2.9k rows
  repeatedly is pure wall-clock cost.

## 3B vs 8B

Defaults to `Llama-3.2-3B`. Predicting a price from a short description is pattern-matching,
not open-ended reasoning, so 8B's extra capacity buys little here while costing ~2.5× the
training time, VRAM, and later Modal inference latency. `MODEL_NAME` is a one-line toggle if
you want to check empirically — train both, push both, and compare real MAE/RMSE on the
held-out test set with `notebooks/04_model_comparison.ipynb`.

## Speed levers deliberately left off

- `packing=True` — TRL's default packing strategy enables `padding_free`, which needs
  FlashAttention 2/3. T4 is Turing; it doesn't support FA2. (`group_by_length=True` is used
  instead — it cuts padding waste with no such requirement.)
- `dataset_num_proc` — tokenization runs at ~1,300 examples/sec, nowhere near the bottleneck.
- Disabling gradient checkpointing — real speed win, but removes the VRAM headroom that keeps
  the 8B toggle viable.

In [1]:
# No -U: on Kaggle, force-upgrading every package independently pulls torch/transformers/peft/
# accelerate to versions that aren't necessarily mutually compatible (confirmed - with -U,
# imports break entirely). Without -U, pip keeps Kaggle's pre-installed, mutually-tested
# torch/transformers/peft/accelerate/datasets as-is, and only freshly installs trl/bitsandbytes
# (not part of the base image), which still resolves to a recent-enough version for everything
# this notebook uses.
!pip install -q torch transformers peft accelerate trl bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 50.1 MB/s eta 0:00:00


In [2]:
# CUDA_VISIBLE_DEVICES MUST be set before torch (or anything importing torch) is imported,
# and only takes effect in a FRESH process. If you've already run cells in this kernel,
# restart it (Run -> Restart Session) and re-run from the top, or this is a silent no-op.
#
# Why: Trainer auto-wraps the model in torch.nn.DataParallel whenever >1 GPU is visible,
# independent of device_map. bitsandbytes' 4-bit layers don't survive DataParallel's
# replication, which caused the CUBLAS_STATUS_EXECUTION_FAILED crash.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from datasets import load_dataset
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Validated here rather than in the final cell, so a bad value fails in seconds instead of
# after hours of training.
HF_USER = "Kuldeep22116048"
assert HF_USER and "YOUR_HF" not in HF_USER, "Set HF_USER to your Hugging Face username."
REPO_NAME = f"{HF_USER}/stealthedeal-price-llama3.2"
print(f"Will push adapter to: {REPO_NAME}\n")

# Print the ACTUAL environment. These libraries change fast and Kaggle's base image moves
# without notice - if something below breaks later, this output is the first thing to check.
import transformers, peft, trl, accelerate, bitsandbytes, datasets as ds_lib
print(f"python       {os.sys.version.split()[0]}")
print(f"torch        {torch.__version__}")
print(f"transformers {transformers.__version__}")
print(f"peft         {peft.__version__}")
print(f"trl          {trl.__version__}")
print(f"accelerate   {accelerate.__version__}")
print(f"bitsandbytes {bitsandbytes.__version__}")
print(f"datasets     {ds_lib.__version__}")
print()
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPUs visible to torch: {torch.cuda.device_count()}  (must be 1 - see note above)")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}  compute capability {props.major}.{props.minor}  "
          f"VRAM {props.total_memory / 1e9:.1f} GB")
    # bf16 needs compute capability >= 8.0 (Ampere+). T4 is 7.5 (Turing), P100 is 6.0 -
    # both fp16-only, which is why this notebook trains in fp16 throughout.
    print(f"bf16 natively supported: {props.major >= 8}  -> training in fp16")

# Login to HF
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

Will push adapter to: Kuldeep22116048/stealthedeal-price-llama3.2

python       3.12.13
torch        2.10.0+cu128
transformers 5.0.0
peft         0.19.1
trl          1.9.2
accelerate   1.13.0
bitsandbytes 0.50.0
datasets     5.0.0

CUDA available: True
GPUs visible to torch: 1  (must be 1 - see note above)
GPU: Tesla T4  compute capability 7.5  VRAM 15.6 GB
bf16 natively supported: False  -> training in fp16


In [3]:
import glob

TRAIN_PATH = '/kaggle/input/datasets/damnyadav/amazon-30k/finetune_train.jsonl'
VAL_PATH   = '/kaggle/input/datasets/damnyadav/amazon-30k/finetune_val.jsonl'

# If the exact path is off (Kaggle mount layouts vary), find the files anywhere under
# /kaggle/input rather than failing - saves a round trip.
def resolve(path, filename):
    if os.path.exists(path):
        return path
    hits = glob.glob(f'/kaggle/input/**/{filename}', recursive=True)
    if not hits:
        raise FileNotFoundError(
            f"{path} not found, and no {filename} anywhere under /kaggle/input. "
            "Check that the dataset is attached to this notebook (right panel -> Input)."
        )
    print(f"NOTE: {path} not found; using discovered path {hits[0]}")
    return hits[0]

TRAIN_PATH = resolve(TRAIN_PATH, 'finetune_train.jsonl')
VAL_PATH   = resolve(VAL_PATH,   'finetune_val.jsonl')

dataset = load_dataset("json", data_files={"train": TRAIN_PATH, "test": VAL_PATH})

# No formatting_func needed: our JSONL rows are already {"prompt": ..., "completion": ...},
# which TRL recognizes natively as prompt-completion format and trains on by computing loss
# ONLY on the completion (the price) - not on the fixed boilerplate question text in the
# prompt. Concatenating them into one "text" blob via formatting_func (the old approach)
# would force full-sequence loss, wasting gradient signal on text that never varies.
train_dataset = dataset['train']

# Subsample the eval set. It's used only to pick the best checkpoint, and evaluating all
# ~2.9k rows at every eval is real wall-clock time repeated many times over the run;
# ~800 rows estimates validation loss closely enough to rank checkpoints.
EVAL_SUBSET = 800
eval_dataset = dataset['test']
if len(eval_dataset) > EVAL_SUBSET:
    eval_dataset = eval_dataset.shuffle(seed=42).select(range(EVAL_SUBSET))

print(f"\ntrain: {len(train_dataset)} rows   eval: {len(eval_dataset)} rows")
print("\nSample row:")
print(train_dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]


train: 23471 rows   eval: 800 rows

Sample row:
{'prompt': 'What does this cost to the nearest rupee?\n\nTitle: Rakhi gift - 6 Chocolates Gift Box - Rakhi with gifts with Rakhi\nCategory: Grocery & Gourmet Foods\nBrand: CHOCOCRAFT\nDescription: This Chocolate Gift Box contains delectable assorted Chocolates. Filled with roasted almonds, fruit & nuts and butter scotch, they will tempt and delight the taste buds. These are packed in a beautiful wooden box especially designed for an elegant presentation. The sturdy box can be reused.\nDetails: 227 Grams\n\nPrice is ₹', 'completion': '595.00'}


In [4]:
# Default: 3B for speed (see markdown above). To compare against the larger base model,
# swap this one line - nothing else in the notebook needs to change:
MODEL_NAME = "meta-llama/Llama-3.2-3B"
# MODEL_NAME = "meta-llama/Llama-3.1-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # fp16 everywhere: T4/P100 have no native bf16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    # Explicit float16. Without this, transformers v5 infers the dtype from the checkpoint's
    # config.json, and Meta's Llama checkpoints declare bfloat16 there.
    dtype=torch.float16,
    # Redundant with CUDA_VISIBLE_DEVICES="0" above, kept as explicit intent.
    device_map={"": 0},
)

# use_cache and gradient checkpointing are mutually exclusive; Trainer would warn and
# disable the cache anyway, so do it explicitly.
model.config.use_cache = False

# use_reentrant=False is required for gradient checkpointing to work with PEFT adapters -
# the legacy reentrant path can produce "none of the output has requires_grad=True" because
# the frozen 4-bit base inputs don't require grad.
model = prepare_model_for_kbit_training(
    model, gradient_checkpointing_kwargs={"use_reentrant": False}
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)

# ---------------------------------------------------------------------------------------
# THE fix for: NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda"
#              not implemented for 'BFloat16'
#
# fp16=True makes Trainer use a GradScaler, and GradScaler's unscale kernel supports only
# float16/float32 gradients - NOT bfloat16. A gradient's dtype follows its parameter's dtype,
# and the only trainable params here are the LoRA adapters created by get_peft_model() above.
# Whether those land in fp32 depends on prepare_model_for_kbit_training()'s upcasting
# behavior, which has changed across PEFT versions (the classic QLoRA recipe relied on an
# unconditional upcast that no longer always happens) - so setting the BASE model's dtype
# doesn't control it.
#
# Rather than depend on any of that, force it: every trainable param becomes fp32, so every
# gradient is fp32, so GradScaler always works. Costs nothing meaningful - LoRA adapters are
# a tiny fraction of params, the base model stays 4-bit, and fp16 autocast still does the
# heavy matmuls at fp16 speed.
# ---------------------------------------------------------------------------------------
recast = 0
for _, param in model.named_parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.to(torch.float32)
        recast += 1

trainable_dtypes = {p.dtype for p in model.parameters() if p.requires_grad}
print(f"Recast {recast} trainable params to float32")
print(f"Trainable param dtypes: {trainable_dtypes}")
# Fail loudly HERE at setup rather than cryptically 20 minutes into training.
assert trainable_dtypes == {torch.float32}, (
    f"Expected all trainable params to be float32, found {trainable_dtypes}. "
    "GradScaler (fp16=True) cannot unscale bfloat16 gradients."
)

model.print_trainable_parameters()

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Recast 0 trainable params to float32
Trainable param dtypes: {torch.float32}
trainable params: 4,587,520 || all params: 3,217,337,344 || trainable%: 0.1426


In [5]:
# =======================================================================================
# DTYPE GUARD - the fix for:
#   NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda"
#                        not implemented for 'BFloat16'
#
# fp16=True makes Trainer use a GradScaler, whose unscale kernel handles only fp16/fp32
# gradients - never bf16. A gradient's dtype follows its PARAMETER's dtype, and the only
# trainable params are the LoRA adapters. Whether those land in fp32 depends on
# prepare_model_for_kbit_training()'s upcasting, which changed across PEFT versions.
#
# This runs HERE, in the training cell, on purpose. It mutates whatever `model` object is
# currently in memory, so it works even if the model cell above wasn't re-run or the model
# was built by an earlier version of that cell. Cheap and idempotent - safe to re-run.
# =======================================================================================
recast = 0
for _, p in model.named_parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.to(torch.float32)
        recast += 1
still_bad = {p.dtype for p in model.parameters() if p.requires_grad} - {torch.float32}
n_trainable = sum(1 for p in model.parameters() if p.requires_grad)
print(f"[dtype guard] {n_trainable} trainable params | recast {recast} -> fp32 | "
      f"remaining non-fp32: {still_bad if still_bad else 'none'}")
assert not still_bad, f"Trainable params still non-fp32: {still_bad}"

# ESCAPE HATCH: if the BFloat16 GradScaler error somehow STILL appears after the guard
# above, set this to False. That turns off fp16 mixed precision entirely, so no GradScaler
# is ever created and the error becomes structurally impossible. Costs ~1.5-2x speed and
# more activation memory - if it then OOMs, drop PER_DEVICE_BATCH_SIZE to 4 and raise
# GRAD_ACCUM_STEPS to 4 (keeping the effective batch size at 16).
USE_FP16 = False

PER_DEVICE_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
NUM_EPOCHS = 3

# warmup_ratio is deprecated (removed in transformers v5.2) in favor of warmup_steps -
# compute it from the real dataset size so it stays correct however many rows survived cleaning.
effective_batch_size = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS
steps_per_epoch = len(train_dataset) // effective_batch_size
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = max(1, int(total_steps * 0.03))
print(f"steps/epoch={steps_per_epoch}  total_steps={total_steps}  warmup_steps={warmup_steps}  "
      f"fp16={USE_FP16}")

training_args = SFTConfig(
    output_dir="./results",
    max_length=512,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    optim="paged_adamw_32bit",
    logging_steps=25,
    learning_rate=2e-4,
    num_train_epochs=NUM_EPOCHS,
    warmup_steps=warmup_steps,
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    group_by_length=True,

    # precision: fp16 or pure fp32, never bf16 (T4/P100 have no native bf16 support)
    fp16=USE_FP16,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},   # required with PEFT adapters

    loss_type="nll",   # default "chunked_nll" monkey-patches forward and breaks on
                       # PeftModel-over-4bit ("functools.partial has no attribute __func__")
    report_to="none",

    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,          # must be a multiple of eval_steps for load_best_model_at_end
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    # No peft_config= : `model` is ALREADY a PeftModel, and passing it again is a hard error.
    processing_class=tokenizer,   # replaces the deprecated tokenizer= argument
    args=training_args,
)

trainer.train()

[dtype guard] 112 trainable params | recast 0 -> fp32 | remaining non-fp32: none
steps/epoch=1466  total_steps=4398  warmup_steps=131  fp16=False


Adding EOS to train dataset:   0%|          | 0/23471 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/23471 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/23471 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/23471 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
500,1.604439,1.467867,1.481697,0.718447,885673.000000
1000,1.504664,1.382985,1.417557,0.728846,1771239.000000
1500,1.195551,1.353295,1.298415,0.735994,2664461.000000
2000,1.245963,1.333886,1.289055,0.740521,3548109.000000
2500,1.245640,1.310548,1.298038,0.743879,4431375.000000
3000,1.241607,1.308473,1.217116,0.743648,5318223.000000
3500,1.202084,1.305106,1.228455,0.745123,6203923.000000
4000,1.197803,1.304878,1.227045,0.744093,7087814.000000


TrainOutput(global_step=4401, training_loss=1.3344733320130893, metrics={'train_runtime': 17292.4679, 'train_samples_per_second': 4.072, 'train_steps_per_second': 0.255, 'total_flos': 1.3281777158500762e+17, 'train_loss': 1.3344733320130893, 'epoch': 3.0})

In [6]:
# HF_USER / REPO_NAME were set and validated back in the imports cell, so this can't fail
# on a placeholder after a long training run.
# load_best_model_at_end=True means trainer.model is the lowest-eval_loss checkpoint here,
# not necessarily the final step's weights.
trainer.model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)   # push the tokenizer too, so the Modal service can load
                                    # everything from this one repo
print(f"Successfully pushed to {REPO_NAME}. You can now deploy this on Modal!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Successfully pushed to Kuldeep22116048/stealthedeal-price-llama3.2. You can now deploy this on Modal!


## Next step: wire this repo into Modal

Copy the printed repo name above (`your-username/stealthedeal-price-llama3.2`) into the
`HF_REPO` constant near the top of `modal_deployments/pricer_service.py` on your local
machine, then run `modal deploy modal_deployments/pricer_service.py`. That's the *only*
line in that file you need to change - see `SETUP.md` for the full Modal setup walkthrough.